# Exercise 1: Audit another language

*Corpora at Scale · version 0.1.0*

> Repeat the hands-on for a second Turkic language in the same corpus. Which contamination type dominates, and how does it compare with the first language?

**Setup.** `pip install "turkicnlp==0.5.0" pyhfst fasttext-wheel huggingface_hub requests`. Open the notebook from the `exercises/` folder: the first code cell imports the chapter's audit functions from `../code/remaining/corpus.py`. GlotLID (`model_v3.bin`, about 1.6 GB) is downloaded from Hugging Face once; set `GLOTLID_PATH` to use a local copy. The Apertium analysers come with TurkicNLP and are read with the pure-Python `pyhfst`. The text comes from the X-WMT test sets on GitHub, so internet access is needed.

Before the audit signals are applied to a real corpus subset, it is worth checking what they do on documents whose composition is known. This notebook builds five kinds of five-sentence documents from the X-WMT test sets: target language only, a close neighbour (Kazakh), Russian, three target sentences plus two Russian ones, and the target converted to the Common Turkic Alphabet. It then runs the chapter's `audit` function with unrestricted GlotLID and the target-language Apertium analyser, first for Kyrgyz and then for Bashkir as the second language. The "kept" column applies an illustrative rule (target share and analyser coverage both at least 0.6); real thresholds have to be chosen on labelled development data.

In [ ]:
import os, re, sys, unicodedata
from collections import Counter
import requests

sys.path.insert(0, os.path.abspath("../code/remaining"))      # the book's corpus.py
from corpus import audit, script_shares, fst_coverage

BASE = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/master/xwmt/test"

def xwmt_side(pair, lang):
    """Non-empty lines of one side of an X-WMT test pair, e.g. xwmt_side("en-ky", "ky")."""
    r = requests.get(f"{BASE}/{pair}/{pair}.{lang}", timeout=30)
    r.raise_for_status()
    return [line.strip() for line in r.text.splitlines() if line.strip()]

def glotlid_model():
    """Unrestricted GlotLID v3 (fastText). Uses GLOTLID_PATH if set, otherwise downloads the model."""
    import fasttext
    path = os.environ.get("GLOTLID_PATH")
    if not path:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
    return fasttext.load_model(path)

def lid_labels(model, text, k=1):
    """Labels, best first. A list input avoids a NumPy 2 incompatibility in fasttext's single-string path."""
    return model.predict([text.replace("\n", " ")], k=k)[0][0]

def load_analyser(lang, script="Cyrl"):
    """Apertium analyser from TurkicNLP, read with pyhfst. lookup(word) -> [(reading, weight), ...]."""
    import pyhfst, turkicnlp
    from turkicnlp.resources.registry import ModelRegistry
    turkicnlp.download(lang, processors=["morph"])
    path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
    fst = pyhfst.HfstInputStream(str(path)).read()
    def lookup(word):
        readings = fst.lookup(word) or fst.lookup(word.lower())
        return [(reading, float(weight)) for reading, weight in readings]
    return lookup

WORD = re.compile(r"[^\W\d_]+")
def words(text):
    return WORD.findall(unicodedata.normalize("NFC", text))

In [ ]:
from statistics import mean
from turkicnlp.scripts import Script
from turkicnlp.scripts.transliterator import Transliterator

model = glotlid_model()
predict = lambda s: lid_labels(model, s)[0]      # full label space, Russian included

def make_documents(target, neighbour, russian, to_cts, n=20, per_doc=5):
    """Documents of known composition from disjoint X-WMT line ranges (one sentence per line)."""
    seg = lambda lines, start: [lines[start + i * per_doc: start + (i + 1) * per_doc] for i in range(n)]
    docs = [("target", s) for s in seg(target, 0)]
    docs += [("neighbour", s) for s in seg(neighbour, 100)]
    docs += [("Russian", s) for s in seg(russian, 200)]
    docs += [("target 3 + Russian 2", t[:3] + r[:2]) for t, r in zip(seg(target, 300), seg(russian, 400))]
    docs += [("target in CTS", [to_cts.transliterate(x) for x in s]) for s in seg(target, 0)]
    return docs

def audit_documents(docs, target_label, lookup):
    rows = []
    for kind, sentences in docs:
        text = "\n".join(sentences)      # here the sentence boundaries are known; in web text they are not
        r = audit(text, sentences, target_label, predict_all=predict, lookup=lookup, words=words(text))
        rows.append({"kind": kind, "target": r["target_share"], "russian": r["russian_share"],
                     "fst": r["fst"]["coverage"], "cyrillic": r["script"]["shares"]["CYRILLIC"]})
    return rows

def summarise(rows, min_target=0.6, min_fst=0.6):
    print(f"{'document type':22} {'target':>7} {'Russian':>8} {'FST cov.':>9} {'Cyrillic':>9} {'kept':>6}")
    for kind in dict.fromkeys(r["kind"] for r in rows):
        rs = [r for r in rows if r["kind"] == kind]
        kept = sum(r["target"] >= min_target and (r["fst"] or 0) >= min_fst for r in rs)
        print(f"{kind:22} {mean(r['target'] for r in rs):7.2f} {mean(r['russian'] for r in rs):8.2f} "
              f"{mean(r['fst'] or 0 for r in rs):9.2f} {mean(r['cyrillic'] or 0 for r in rs):9.2f} {kept:>3}/{len(rs)}")

In [ ]:
ky, kk, ru = xwmt_side("en-ky", "ky"), xwmt_side("en-kk", "kk"), xwmt_side("en-ru", "ru")
kir_docs = make_documents(ky, kk, ru, Transliterator("kir", Script.CYRILLIC, Script.COMMON_TURKIC))
kir_rows = audit_documents(kir_docs, "__label__kir_Cyrl", load_analyser("kir"))
summarise(kir_rows)
# Which labels did the neighbour documents get?
print(Counter(predict(s) for kind, sents in kir_docs if kind == "neighbour" for s in sents).most_common(3))
print(Counter(predict(s) for kind, sents in kir_docs if kind == "target in CTS" for s in sents).most_common(3))

In [ ]:
ba = xwmt_side("en-ba", "ba")
bak_docs = make_documents(ba, kk, ru, Transliterator("bak", Script.CYRILLIC, Script.COMMON_TURKIC))
bak_rows = audit_documents(bak_docs, "__label__bak_Cyrl", load_analyser("bak"))
summarise(bak_rows)
print(Counter(predict(s) for kind, sents in bak_docs if kind == "target in CTS" for s in sents).most_common(3))

## Worked solution

Output of `experiments/ch04_exercises/run_ch04_exercises.py` on 18 October 2026, UTC (GlotLID v3, TurkicNLP 0.5.0 Apertium analysers, X-WMT test sets; 20 documents per type):

```
Kyrgyz (target __label__kir_Cyrl)
document type           target  Russian  FST cov.  Cyrillic   kept
target                    1.00     0.00      0.83      0.98  20/20
neighbour                 0.00     0.00      0.28      0.97   0/20
Russian                   0.00     1.00      0.09      0.97   0/20
target 3 + Russian 2      0.60     0.40      0.52      0.97   4/20
target in CTS             0.00     0.00      0.00      0.00   0/20
[('__label__kaz_Cyrl', 99), ('__label__kaa_Cyrl', 1)]
[('__label__tur_Latn', 49), ('__label__gag_Latn', 33), ('__label__crh_Latn', 7)]

Bashkir (target __label__bak_Cyrl)
document type           target  Russian  FST cov.  Cyrillic   kept
target                    1.00     0.00      0.83      0.98  20/20
neighbour                 0.00     0.00      0.21      0.97   0/20
Russian                   0.00     1.00      0.20      0.97   0/20
target 3 + Russian 2      0.60     0.40      0.58      0.96   8/20
target in CTS             0.00     0.00      0.00      0.00   0/20
[('__label__tat_Latn', 91), ('__label__crh_Latn', 4), ('__label__azj_Latn', 2)]
```

**Reading the tables.** For both languages, unrestricted GlotLID labels every sentence of the target, neighbour and Russian documents correctly (the neighbour sentences get `kaz_Cyrl`, 99 of 100 for Kyrgyz), so the target share alone separates these three types here. The X-WMT sentences are long, edited news text, which makes identification easier than on web text. The script share does not help: Russian documents are as Cyrillic as the target ones (0.97). Analyser coverage is lower and less clear-cut: 0.83 on target documents, but 0.28 (Kyrgyz analyser) and 0.21 (Bashkir analyser) on Kazakh documents and 0.09 and 0.20 on Russian ones, because shared words and names are recognised. On these documents, coverage separates the neighbour less well than language identification does.

The CTS documents get no target label at all: GlotLID assigns Kyrgyz in CTS mostly to Turkish, Gagauz or Crimean Tatar, and Bashkir in CTS to Latin-script Tatar. This is why language identification comes before conversion in the chapter's pipeline. The mixed documents show that a document-level rule encodes an inclusion policy: with three of five sentences in the target language, the illustrative rule keeps 4 of 20 Kyrgyz documents but 8 of 20 Bashkir ones, a difference that comes only from how many Russian words each analyser recognises.

**Comparing the two languages.** The signals behave alike for Kyrgyz and Bashkir. The clearest difference is that the Bashkir analyser recognises more Russian words (0.20 against 0.09), so analyser coverage is a weaker filter against Russian for Bashkir than for Kyrgyz. These are fixture results on documents of known composition; they say nothing about which contamination type dominates in a real subset.

**On a real corpus subset**, the dominant contamination type is the type with the largest human-labelled share in a uniform sample, not the type the signals flag most often. Build the sample with `reservoir` (hands-on), label it, and report each type with its confidence interval. With a known-composition check like this one, you can also say which types the automatic signals would miss.

## Your turn — extension

1. Replace the X-WMT documents by a uniform sample of a real subset, for example FineWeb2 `kir_Cyrl`, streamed with the `datasets` library, and label 100 documents by hand before looking at the signals.
2. Add a document type for another neighbour (Tatar for Bashkir, Uzbek for Kyrgyz) and see whether analyser coverage separates it.
3. Change `min_target` and `min_fst`, and note how many target documents you lose for each Russian or neighbour document you remove.